<a href="https://colab.research.google.com/github/gamernipung10157-star/calculator/blob/main/ann_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

ds = load_dataset("allenai/openbookqa", "main")

In [ ]:
# Convert to pandas DataFrame for easy exploration
import pandas as pd

# Assuming there is a 'train' split, let's look at it
split_name = list(ds.keys())[0]
df = pd.DataFrame(ds[split_name])
display(df.head())

,id,question_stem,choices,answerKey
0,7-980,The sun is responsible for,"{'text': ['puppies learning new tricks', 'chil...",D
1,7-584,When standing miles away from Mount Rushmore,"{'text': ['the mountains seem very close', 'th...",D
2,7-870,When food is reduced in the stomach,"{'text': ['the mind needs time to digest', 'ta...",C
3,7-321,Stars are,"{'text': ['warm lights that float', 'made out ...",C
4,9-732,You can make a telescope with a,"{'text': ['straw', 'Glass', 'Candle', 'mailing...",D


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# OpenBookQA uses 'question_stem' as the primary text field instead of 'instruction'
instruction_lengths = df['question_stem'].astype(str).str.len()

# Split into 3 difficulty tiers based on text length quantiles
low_threshold = instruction_lengths.quantile(0.33)
high_threshold = instruction_lengths.quantile(0.66)

def assign_difficulty(text):
    length = len(str(text))
    if length < low_threshold:
        return 'easy'
    elif length < high_threshold:
        return 'medium'
    else:
        return 'hard'

df['difficulty'] = df['question_stem'].apply(assign_difficulty)

# Map difficulties to numeric values
difficulty_map = {'easy': 0.1, 'medium': 0.5, 'hard': 0.9}
df['difficulty_numeric'] = df['difficulty'].map(lambda x: difficulty_map.get(str(x).lower(), 0.5))

# Generate synthetic user play data matched to the new dataset structure
np.random.seed(42)
num_simulations = 5000

sim_student_skill = np.random.uniform(0.1, 0.9, num_simulations)
sim_question_difficulty = np.random.choice(df['difficulty_numeric'], num_simulations)
sim_recent_success = np.random.uniform(0.0, 1.0, num_simulations)

# Sigmoid relation for correct answers
logit = 5.0 * (sim_student_skill - sim_question_difficulty) + 1.5 * (sim_recent_success - 0.5)
prob_correct = 1 / (1 + np.exp(-logit))
sim_correct = (np.random.rand(num_simulations) < prob_correct).astype(int)

# Response times
sim_response_time = np.clip(15 * (sim_question_difficulty / (sim_student_skill + 0.1)) + np.random.normal(5, 2, num_simulations), 2, 45)

history_df = pd.DataFrame({
    'student_skill': sim_student_skill,
    'question_difficulty': sim_question_difficulty,
    'recent_success_rate': sim_recent_success,
    'response_time': sim_response_time,
    'is_correct': sim_correct
})

print("Generated synthetic player interaction profiles successfully based on OpenBookQA question stem complexity!")

Generated synthetic player interaction profiles successfully based on OpenBookQA question stem complexity!


In [ ]:
# Prepare data for ANN
X = history_df[['student_skill', 'question_difficulty', 'recent_success_rate', 'response_time']].values
y = history_df['is_correct'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define Neural Network architecture
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train_scaled, y_train, epochs=15, batch_size=64, validation_split=0.1, verbose=1)

# Evaluate accuracy
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nANN Model Trained. Test Accuracy: {accuracy * 100:.2f}%")

Epoch 1/15


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7428 - loss: 0.5651 - val_accuracy: 0.7400 - val_loss: 0.5266
Epoch 2/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7828 - loss: 0.4704 - val_accuracy: 0.7550 - val_loss: 0.4929
Epoch 3/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7925 - loss: 0.4513 - val_accuracy: 0.7525 - val_loss: 0.4858
Epoch 4/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7917 - loss: 0.4436 - val_accuracy: 0.7575 - val_loss: 0.4792
Epoch 5/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7975 - loss: 0.4370 - val_accuracy: 0.7700 - val_loss: 0.4812
Epoch 6/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7936 - loss: 0.4406 - val_accuracy: 0.7700 - val_loss: 0.4795
Epoch 7/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7883 - loss: 0.4381 - val_accuracy: 0.7700 - val_loss: 0.4787
Epoch 8/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7944 - loss: 0.4354 - val_accuracy: 0.7725 - val_loss: 0.4786
Epo

In [ ]:
import time
import random
import numpy as np

def clean_text(text):
    if not isinstance(text, str):
        return str(text)
    # Strip LaTeX delimiters and clean spacing
    cleaned = text.replace('$$', '').replace('$', '')
    return cleaned.strip()

def run_educational_battle(questions_df, predictor_model, scaler):
    print("=======================================================")
    print("⚔️ WELCOME TO THE ADAPTIVE EDUCATIONAL ARENA ⚔️")
    print("=======================================================")

    # Boss roster
    bosses = {
        "1": {"name": "Goblin Recruit", "hp": 60, "desc": "An easy warm-up opponent.", "emoji": "👺"},
        "2": {"name": "Shadow Assassin", "hp": 120, "desc": "A standard martial challenge.", "emoji": "👤"},
        "3": {"name": "Cursed Golem", "hp": 200, "desc": "A sturdy tank with high vitality.", "emoji": "🗿"},
        "4": {"name": "Elder Dragon", "hp": 350, "desc": "An absolute raid boss for masters.", "emoji": "🐉"}
    }

    print("Choose your opponent:")
    for key, boss in bosses.items():
        print(f"  {key}) {boss['emoji']} {boss['name']} (HP: {boss['hp']}) - {boss['desc']}")

    boss_choice = input("Select a boss (1-4, default 2): ").strip()
    selected_boss = bosses.get(boss_choice, bosses["2"])

    print(f"\n🔥 You have challenged {selected_boss['emoji']} {selected_boss['name']}! 🔥")
    print("Solve science questions to attack the Boss. Speed and correctness matter!\n")

    # Init characters
    player_hp = 100
    boss_hp = selected_boss["hp"]
    max_boss_hp = selected_boss["hp"]
    player_skill = 0.5
    recent_answers = []

    round_num = 1

    while player_hp > 0 and boss_hp > 0:
        print(f"--- ROUND {round_num} ---")
        print(f"❤️ Player HP: {player_hp}/100 | {selected_boss['emoji']} {selected_boss['name']} HP: {boss_hp}/{max_boss_hp}")

        # ANN recommendations
        recent_success_rate = sum(recent_answers) / len(recent_answers) if recent_answers else 0.5
        diff_candidates = [0.1, 0.5, 0.9]
        predicted_probabilities = []
        for diff in diff_candidates:
            test_profile = np.array([[player_skill, diff, recent_success_rate, 10.0]])
            test_profile_scaled = scaler.transform(test_profile)
            pred = predictor_model.predict(test_profile_scaled, verbose=0)[0][0]
            predicted_probabilities.append(pred)

        target_prob = 0.65
        best_diff_idx = np.argmin([abs(p - target_prob) for p in predicted_probabilities])
        ann_recommended_diff = diff_candidates[best_diff_idx]
        diff_names = {0.1: 'easy', 0.5: 'medium', 0.9: 'hard'}
        ann_rec_name = diff_names[ann_recommended_diff].upper()

        # Prompt user to choose difficulty
        print(f"\n[ANN Recommendation: {ann_rec_name} (Est. Success: {predicted_probabilities[best_diff_idx]:.2f})]")
        print("Choose question difficulty:")
        print("  1) Easy (Short problems)")
        print("  2) Medium (Mid-length problems)")
        print("  3) Hard (Complex word problems)")
        print("  4) Let ANN choose for me")

        choice = input("Select an option (1-4): ").strip()
        if choice == '1':
            selected_diff_val = 0.1
        elif choice == '2':
            selected_diff_val = 0.5
        elif choice == '3':
            selected_diff_val = 0.9
        else:
            selected_diff_val = ann_recommended_diff
            print(f"\n[ANN selected {diff_names[selected_diff_val].upper()} for you]")

        difficulty_tag = diff_names[selected_diff_val]

        # Filter questions matching difficulty safely
        candidates = questions_df[questions_df['difficulty'].astype(str).str.lower() == difficulty_tag]
        if candidates.empty:
            candidates = questions_df

        # Select random question
        question_row = candidates.sample(n=1).iloc[0]

        # Clean and display question stem
        cleaned_question = clean_text(question_row['question_stem'])
        print(f"\nQuestion ({difficulty_tag.upper()}):\n{cleaned_question}\n")

        # Print multiple choice options if present
        choices_dict = question_row.get('choices', {})
        correct_answer_text = ""
        if isinstance(choices_dict, dict) and 'label' in choices_dict and 'text' in choices_dict:
            labels = choices_dict['label']
            texts = choices_dict['text']
            for label, text in zip(labels, texts):
                print(f"  {label}) {clean_text(text)}")
            correct_answer_text = str(question_row.get('answerKey', '')).strip()
        else:
            correct_answer_text = clean_text(str(question_row.get('answerKey', '')).strip())

        print("\nType your answer (usually A, B, C, or D):")
        start_time = time.time()
        user_answer = input("\nYour Answer: ").strip()
        end_time = time.time()

        is_correct = 1 if (user_answer.lower() == correct_answer_text.lower() and len(user_answer) > 0) else 0

        response_time = end_time - start_time
        print(f"Time taken: {response_time:.2f} seconds")
        recent_answers.append(is_correct)
        if len(recent_answers) > 5:
            recent_answers.pop(0)

        if is_correct:
            player_skill = min(1.0, player_skill + 0.05 * (1.1 - selected_diff_val))
            base_damage = random.randint(15, 35)
            speed_multiplier = max(0.5, 1.5 - (response_time / 30.0))
            damage = int(base_damage * (1 + selected_diff_val) * speed_multiplier)

            boss_hp -= damage
            print(f"\n✨ CORRECT! You deal a massive {damage} DMG to {selected_boss['name']}!")
        else:
            player_skill = max(0.0, player_skill - 0.05 * selected_diff_val)
            outcome = random.choice(["dodge", "parry", "graze"])

            if outcome == "dodge":
                print(f"\n❃ INCORRECT!")
                print(f"💨 {selected_boss['name']} DODGED your attack completely! You dealt 0 DMG.")
            elif outcome == "parry":
                print(f"\n❃ INCORRECT!")
                print(f"🛡️ {selected_boss['name']} PARRIED your strike! You dealt 0 DMG.")
            else:
                low_dmg = random.randint(3, 8)
                boss_hp -= low_dmg
                print(f"\n❃ INCORRECT!")
                print(f"⚔️ Your strike grazed {selected_boss['name']}, dealing a minor {low_dmg} DMG!")

            # Boss strikes back!
            boss_damage = int(random.randint(10, 20) * (1 + selected_diff_val))
            player_hp -= boss_damage
            print(f"👿 {selected_boss['name']} counterattacks for {boss_damage} DMG!")
            print(f"The correct answer was: {correct_answer_text}")

        print(f"\nEstimated Player Skill: {player_skill:.2f} | Current Success Rate (Last 5): {recent_success_rate:.2f}")
        print("-" * 40 + "\n")
        round_num += 1

    if boss_hp <= 0:
        print(f"🎉 VICTORY! You defeated {selected_boss['name']}!")
    else:
        print(f"💀 DEFEATED! {selected_boss['name']} has bested you. Try again to improve your skills.")

In [ ]:
import time
import random
import numpy as np
import ipywidgets as widgets
from IPython.display import display

def run_gui_battle(questions_df, predictor_model, scaler):
    state = {
        'player_hp': 100,
        'boss_hp': 120,
        'max_boss_hp': 120,
        'boss_name': "Shadow Assassin",
        'boss_emoji': "👤",
        'player_skill': 0.5,
        'recent_answers': [],
        'current_question': None,
        'correct_answer': "",
        'start_time': 0.0,
        'selected_diff_val': 0.5,
        'round_num': 1
    }

    bosses = {
        "👺 Goblin Recruit (HP: 60)": {"name": "Goblin Recruit", "hp": 60, "emoji": "👺"},
        "👤 Shadow Assassin (HP: 120)": {"name": "Shadow Assassin", "hp": 120, "emoji": "👤"},
        "🗿 Cursed Golem (HP: 200)": {"name": "Cursed Golem", "hp": 200, "emoji": "🗿"},
        "🐉 Elder Dragon (HP: 350)": {"name": "Elder Dragon", "hp": 350, "emoji": "🐉"}
    }

    def get_header_html():
        return """
        <div style='background: #34495e; padding: 12px; border-radius: 6px; text-align: center; color: white; font-family: sans-serif; margin-bottom: 10px;'>
            <h2 style='margin: 0; font-size: 18px;'>⚔️ ADAPTIVE EDUCATIONAL ARENA ⚔️</h2>
            <p style='margin: 4px 0 0 0; font-size: 12px; opacity: 0.8;'>Defeat bosses using science • Powered by ANN</p>
        </div>
        """

    def get_battle_hud_html():
        player_pct = max(0, min(100, state['player_hp']))
        boss_pct = max(0, min(100, int((state['boss_hp'] / state['max_boss_hp']) * 100)))
        return f"""
        <div style='display: flex; justify-content: space-between; gap: 10px; font-family: sans-serif;'>
            <div style='flex: 1; border: 1px solid #bdc3c7; padding: 10px; border-radius: 6px; background: #fafbfc;'>
                <h4 style='margin: 0 0 4px 0; color: #2c3e50;'>🛡️ Hero (Player)</h4>
                <div style='background: #ecf0f1; border-radius: 4px; height: 10px; overflow: hidden; margin-bottom: 4px;'>
                    <div style='background: #2ecc71; width: {player_pct}%; height: 100%;'></div>
                </div>
                <span style='font-size: 12px; font-weight: bold; color: #27ae60;'>HP: {state['player_hp']}/100</span>
                <span style='font-size: 11px; color: #7f8c8d; margin-left: 8px;'>Skill: {state['player_skill']:.2f}</span>
            </div>
            <div style='flex: 1; border: 1px solid #bdc3c7; padding: 10px; border-radius: 6px; background: #fafbfc;'>
                <h4 style='margin: 0 0 4px 0; color: #2c3e50;'>{state['boss_emoji']} {state['boss_name']}</h4>
                <div style='background: #ecf0f1; border-radius: 4px; height: 10px; overflow: hidden; margin-bottom: 4px;'>
                    <div style='background: #e74c3c; width: {boss_pct}%; height: 100%;'></div>
                </div>
                <span style='font-size: 12px; font-weight: bold; color: #c0392b;'>HP: {state['boss_hp']}/{state['max_boss_hp']}</span>
                <span style='font-size: 11px; color: #7f8c8d; margin-left: 8px;'>Round: {state['round_num']}</span>
            </div>
        </div>
        """

    header_output = widgets.HTML(value=get_header_html())
    hud_output = widgets.HTML(value="")
    feedback_output = widgets.HTML(value="<div style='text-align: center; color: #7f8c8d; font-family: sans-serif; padding: 12px; border: 1px dashed #cbd5e1; border-radius: 6px;'>Select opponent to begin.</div>")
    question_output = widgets.HTML(value="")
    answer_buttons_layout = widgets.VBox([])

    boss_selector = widgets.Dropdown(
        options=list(bosses.keys()),
        value="👤 Shadow Assassin (HP: 120)",
        description="Target Boss:",
        style={'description_width': 'initial'}
    )

    start_game_btn = widgets.Button(
        description="⚔️ Start Battle",
        layout=widgets.Layout(width='140px', height='32px')
    )

    config_panel = widgets.HBox([boss_selector, start_game_btn], layout=widgets.Layout(justify_content='center', margin='10px 0'))

    diff_selector = widgets.ToggleButtons(
        options=['Easy', 'Medium', 'Hard', 'Let ANN Decide'],
        value='Let ANN Decide',
        description='Difficulty:',
        style={'button_width': '100px'}
    )
    ann_recommendation_label = widgets.HTML(value="")
    gameplay_control_panel = widgets.VBox([ann_recommendation_label, diff_selector], layout=widgets.Layout(visibility='hidden', height='0px'))

    game_layout = widgets.VBox([
        header_output,
        config_panel,
        hud_output,
        gameplay_control_panel,
        question_output,
        answer_buttons_layout,
        feedback_output
    ], layout=widgets.Layout(max_width='680px', border='1px solid #bdc3c7', padding='12px', border_radius='6px'))

    def clean_text(text):
        if not isinstance(text, str): return str(text)
        return text.replace('$$', '').replace('$', '').strip()

    def generate_ann_recommendation():
        recent_success_rate = sum(state['recent_answers']) / len(state['recent_answers']) if state['recent_answers'] else 0.5
        diff_candidates = [0.1, 0.5, 0.9]
        predicted_probabilities = []

        for diff in diff_candidates:
            test_profile = np.array([[state['player_skill'], diff, recent_success_rate, 10.0]])
            test_profile_scaled = scaler.transform(test_profile)
            pred = predictor_model.predict(test_profile_scaled, verbose=0)[0][0]
            predicted_probabilities.append(pred)

        best_diff_idx = np.argmin([abs(p - 0.65) for p in predicted_probabilities])
        ann_recommended_diff = diff_candidates[best_diff_idx]
        diff_names = {0.1: 'EASY', 0.5: 'MEDIUM', 0.9: 'HARD'}

        rec_text = f"💡 <b>ANN Recommendation:</b> {diff_names[ann_recommended_diff]} (Est. Success: {predicted_probabilities[best_diff_idx]*100:.0f}%)"
        ann_recommendation_label.value = f"<div style='font-family: sans-serif; font-size: 12px; margin-bottom: 6px; padding: 6px; background: #f8fafc; border-left: 3px solid #7f8c8d; border-radius: 4px;'>{rec_text}</div>"
        return ann_recommended_diff

    def load_next_turn():
        if state['player_hp'] <= 0 or state['boss_hp'] <= 0:
            check_game_over()
            return

        hud_output.value = get_battle_hud_html()
        ann_val = generate_ann_recommendation()

        choice = diff_selector.value
        if choice == 'Easy':
            state['selected_diff_val'] = 0.1
        elif choice == 'Medium':
            state['selected_diff_val'] = 0.5
        elif choice == 'Hard':
            state['selected_diff_val'] = 0.9
        else:
            state['selected_diff_val'] = ann_val

        diff_names_rev = {0.1: 'easy', 0.5: 'medium', 0.9: 'hard'}
        difficulty_tag = diff_names_rev[state['selected_diff_val']]

        candidates = questions_df[questions_df['difficulty'].astype(str).str.lower() == difficulty_tag]
        if candidates.empty:
            candidates = questions_df

        row = candidates.sample(n=1).iloc[0]
        state['current_question'] = row

        cleaned_question = clean_text(row['question_stem'])
        question_output.value = f"""
        <div style='background: #fafbfc; border-left: 4px solid #7f8c8d; padding: 12px; border-radius: 4px; font-family: sans-serif; margin-top: 8px;'>
            <span style='background: #7f8c8d; color: white; padding: 2px 6px; border-radius: 4px; font-size: 10px; font-weight: bold; text-transform: uppercase;'>{difficulty_tag}</span>
            <p style='margin: 6px 0 0 0; font-size: 14px; color: #2c3e50;'>{cleaned_question}</p>
        </div>
        """

        choices_dict = row.get('choices', {})
        state['correct_answer'] = str(row.get('answerKey', '')).strip().upper()

        buttons_list = []
        if isinstance(choices_dict, dict) and 'label' in choices_dict and 'text' in choices_dict:
            for label, val in zip(choices_dict['label'], choices_dict['text']):
                btn = widgets.Button(
                    description=f"{label}: {clean_text(val)}",
                    layout=widgets.Layout(width='100%', margin='3px 0', height='32px'),
                )
                btn.on_click(make_answer_handler(label))
                buttons_list.append(btn)
        else:
            for letter in ['A', 'B', 'C', 'D']:
                btn = widgets.Button(description=letter, layout=widgets.Layout(width='80px', margin='3px'))
                btn.on_click(make_answer_handler(letter))
                buttons_list.append(btn)

        answer_buttons_layout.children = buttons_list
        state['start_time'] = time.time()

    def on_difficulty_changed(change):
        if change['name'] == 'value':
            # Load next question with new difficulty immediately if game is active
            if state['player_hp'] > 0 and state['boss_hp'] > 0 and gameplay_control_panel.layout.visibility == 'visible':
                load_next_turn()

    diff_selector.observe(on_difficulty_changed)

    def make_answer_handler(answer_char):
        def on_click(b):
            for child in answer_buttons_layout.children:
                child.disabled = True

            response_time = time.time() - state['start_time']
            correct = (answer_char.upper() == state['correct_answer'])
            state['recent_answers'].append(1 if correct else 0)
            if len(state['recent_answers']) > 5:
                state['recent_answers'].pop(0)

            if correct:
                state['player_skill'] = min(1.0, state['player_skill'] + 0.05 * (1.1 - state['selected_diff_val']))
                base_damage = random.randint(15, 30)
                speed_mult = max(0.5, 1.5 - (response_time / 30.0))
                damage = int(base_damage * (1 + state['selected_diff_val']) * speed_mult)
                state['boss_hp'] = max(0, state['boss_hp'] - damage)

                feedback_html = f"""
                <div style='background: #f4fbf7; border: 1px solid #2ecc71; padding: 10px; border-radius: 4px; font-family: sans-serif; margin-top: 8px; color: #27ae60; font-size: 13px;'>
                    <b>Correct (Option {answer_char})</b> | You dealt {damage} DMG in {response_time:.1f}s.
                </div>
                """
            else:
                state['player_skill'] = max(0.0, state['player_skill'] - 0.05 * state['selected_diff_val'])
                outcome = random.choice(["dodge", "parry", "graze"])
                boss_dmg = int(random.randint(10, 18) * (1 + state['selected_diff_val']))
                state['player_hp'] = max(0, state['player_hp'] - boss_dmg)

                if outcome == "dodge":
                    combat_log = f"{state['boss_name']} dodged the attack."
                elif outcome == "parry":
                    combat_log = f"{state['boss_name']} parried the strike."
                else:
                    graze_dmg = random.randint(3, 7)
                    state['boss_hp'] = max(0, state['boss_hp'] - graze_dmg)
                    combat_log = f"Strike grazed {state['boss_name']} for {graze_dmg} DMG."

                feedback_html = f"""
                <div style='background: #fdf2f2; border: 1px solid #e74c3c; padding: 10px; border-radius: 4px; font-family: sans-serif; margin-top: 8px; color: #c0392b; font-size: 13px;'>
                    <b>Incorrect ({answer_char})</b> | Correct answer: {state['correct_answer']}<br>
                    <i>{combat_log}</i> | taken {boss_dmg} DMG!
                </div>
                """

            feedback_output.value = feedback_html
            state['round_num'] += 1

            # Load next turn immediately to avoid blocking with time.sleep
            load_next_turn()
        return on_click

    def check_game_over():
        hud_output.value = get_battle_hud_html()
        answer_buttons_layout.children = []
        gameplay_control_panel.layout.visibility = 'hidden'
        gameplay_control_panel.layout.height = '0px'
        question_output.value = ""

        if state['player_hp'] <= 0:
            feedback_output.value = f"""
            <div style='background: #fdf2f2; border: 1px dashed #e74c3c; text-align: center; padding: 15px; border-radius: 6px; font-family: sans-serif; margin-top: 8px;'>
                <h3 style='color: #c0392b; margin: 0; font-size: 16px;'>DEFEATED</h3>
                <p style='color: #7f8c8d; margin-top: 4px; font-size: 13px;'>{state['boss_name']} has defeated you.</p>
            </div>
            """
        else:
            feedback_output.value = f"""
            <div style='background: #f4fbf7; border: 1px dashed #2ecc71; text-align: center; padding: 15px; border-radius: 6px; font-family: sans-serif; margin-top: 8px;'>
                <h3 style='color: #27ae60; margin: 0; font-size: 16px;'>VICTORY</h3>
                <p style='color: #7f8c8d; margin-top: 4px; font-size: 13px;'>You have defeated {state['boss_name']}!</p>
            </div>
            """
        config_panel.layout.visibility = 'visible'

    def on_start_clicked(b):
        boss_data = bosses[boss_selector.value]
        state['boss_name'] = boss_data['name']
        state['boss_hp'] = boss_data['hp']
        state['max_boss_hp'] = boss_data['hp']
        state['boss_emoji'] = boss_data['emoji']
        state['player_hp'] = 100
        state['round_num'] = 1
        state['recent_answers'] = []
        state['player_skill'] = 0.5

        config_panel.layout.visibility = 'hidden'
        gameplay_control_panel.layout.visibility = 'visible'
        gameplay_control_panel.layout.height = 'auto'
        feedback_output.value = "<div style='text-align: center; color: #7f8c8d; font-size: 13px; margin-top: 8px;'>Loading...</div>"

        load_next_turn()

    start_game_btn.on_click(on_start_clicked)
    display(game_layout)

In [ ]:
# Run the newly designed GUI Battle Engine
run_gui_battle(df, model, scaler)